# 12 - What the evidence allows

Before the reader gets a better chance, how much does the evidence it is shown allow? A classifier is trained only on what the K1 prompt shows: the summary line and the largest transactions, at most 30. It uses the folds and the threshold rule of notebook 08, and its decisions are compared with the LLM in K1 on the 203 accounts of notebook 11.

The classifier learns from the labels of about 309,000 accounts in each round (three folds) and the LLM from none, so this is a ceiling for the information, not a target for the LLM. If the ceiling is low, a better prompt cannot help much and the evidence itself must change.

This notebook was written after the results of notebook 11 were seen. It is exploratory.

In [1]:
%%time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.features import ACH, OTHER_FORMATS
from skeptical_second_reader.model import fit_xgboost, precision_threshold
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.second_reader import (
    GROUP_NAMES,
    K1,
    MAX_ROWS,
    SUSPICIOUS,
)
from skeptical_second_reader.split import (
    N_FOLDS,
    cached_folds,
    cv_predict,
    split_groups,
)
from skeptical_second_reader.transactions import account_positions, transaction_table

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
members = pattern_members(read_patterns(PATTERNS_TXT))
groups = split_groups(owners, members)
reading = pd.read_parquet(DATA_INTERIM / "transaction_reading.parquet")
accounts = reading.index
y = labels.loc[accounts]
fold_of = cached_folds(labels, groups, DATA_INTERIM).loc[accounts]
table = transaction_table(df)
del df
sender, receiver = account_positions(table, accounts)
print(f"accounts {len(y):,}  positives {y.sum():,}  transactions {len(table):,}")

accounts 515,078  positives 6,347  transactions 5,077,237
CPU times: total: 28.2 s
Wall time: 26.5 s


## 1. The evidence as features

Each account's transactions as the prompt lists them: a transaction to itself once, the others once per side. The summary counts all of them, as the summary line does. The shown rows are the largest by dollar amount, ties in table order, as in `account_evidence`. The pair count is the number of transactions in the same direction with the same counterparty, the last column of the prompt table.

In [2]:
%%time
SENT, RECEIVED, SELF = 0, 1, 2
n_accounts = len(accounts)
is_self = sender == receiver
row_of = np.arange(len(table))
side = pd.DataFrame(
    {
        "account": np.concatenate([sender, receiver[~is_self]]),
        "other": np.concatenate([receiver, sender[~is_self]]),
        "row": np.concatenate([row_of, row_of[~is_self]]),
        "direction": np.concatenate(
            [np.where(is_self, SELF, SENT), np.full((~is_self).sum(), RECEIVED)]
        ),
    }
)
side = side.sort_values("row", kind="stable").reset_index(drop=True)
rows = side["row"].to_numpy()
paid = table["paid_currency"].to_numpy()
got = table["received_currency"].to_numpy()
changes = paid != got
fmt = table["payment_format"].to_numpy()
side["usd"] = table["usd"].to_numpy()[rows]
pair = table.groupby(["sender", "receiver"])["usd"].transform("size").to_numpy()
acc = side["account"].to_numpy()
direction = side["direction"].to_numpy()


def count(mask: np.ndarray, weights: np.ndarray | None = None) -> np.ndarray:
    return np.bincount(acc[mask], weights=weights, minlength=n_accounts)


everything = np.ones(len(side), dtype=bool)
sent, received, own = direction == SENT, direction == RECEIVED, direction == SELF
summary = {
    "transactions": count(everything),
    "sent": count(sent),
    "received": count(received),
    "to_itself": count(own),
    "conversions": count(own & changes[rows]),
    "sent_usd": count(sent, table["usd"].to_numpy()[rows][sent]),
    "received_usd": count(received, table["usd_received"].to_numpy()[rows][received]),
}
others = side.loc[~own, ["account", "other"]].drop_duplicates()
summary["counterparties"] = np.bincount(others["account"], minlength=n_accounts)
for name in [ACH, *OTHER_FORMATS]:
    summary[f"format_{name}"] = count(fmt[rows] == name)
summary = pd.DataFrame(summary, index=accounts)

side["rank"] = side.groupby("account")["usd"].rank(method="first", ascending=False)
shown = side[side["rank"] <= MAX_ROWS].copy()
shown_rows = shown["row"].to_numpy()
shown["pair"] = pair[shown_rows]
shown["changes"] = changes[shown_rows]
shown["currency"] = paid[shown_rows]
shown["time"] = table["timestamp"].to_numpy()[shown_rows]
shown["hour"] = shown["time"].dt.floor("h")
shown["is_sent"] = shown["direction"] == SENT
shown["is_received"] = shown["direction"] == RECEIVED
by = shown.groupby("account")
span = by["time"].max() - by["time"].min()
row_features = pd.DataFrame(
    {
        "shown_rows": by.size(),
        "shown_max_usd": by["usd"].max(),
        "shown_median_usd": by["usd"].median(),
        "shown_max_pair": by["pair"].max(),
        "shown_mean_pair": by["pair"].mean(),
        "shown_share_sent": by["is_sent"].mean(),
        "shown_share_received": by["is_received"].mean(),
        "shown_share_changes": by["changes"].mean(),
        "shown_currencies": by["currency"].nunique(),
        "shown_counterparties": shown[shown["direction"] != SELF]
        .groupby("account")["other"]
        .nunique(),
        "shown_span_hours": span.dt.total_seconds() / 3600,
        "shown_most_in_one_hour": shown.groupby(["account", "hour"])
        .size()
        .groupby(level=0)
        .max(),
    }
)
row_features.index = accounts[row_features.index]
row_features = row_features.reindex(accounts)
row_features["shown_counterparties"] = row_features["shown_counterparties"].fillna(0)
X = pd.concat([summary, row_features], axis=1).astype("float32")
assert X["shown_rows"].notna().all()
assert (X["shown_rows"] == np.minimum(X["transactions"], MAX_ROWS)).all()
SUMMARY = list(summary.columns)
ROWS = list(row_features.columns)
del side, shown, others
print(f"features: summary {len(SUMMARY)}, rows {len(ROWS)}")
print(
    "accounts with more transactions than shown:",
    int((X["transactions"] > MAX_ROWS).sum()),
)

features: summary 15, rows 12
accounts with more transactions than shown: 107759
CPU times: total: 14.1 s
Wall time: 14.1 s


## 2. The ceiling on all accounts

Test AUC-PR per round, and the flags of the threshold rule (lowest score with validation precision at least 0.7). For reference, notebook 08: the transaction model 0.4806 +- 0.0260 with all its features, the account model 0.3639 +- 0.0153.

In [3]:
%%time
ceilings = {}
for name, columns in {"summary": SUMMARY, "summary and rows": SUMMARY + ROWS}.items():
    cv = cv_predict(X[columns], y, fold_of, fit_xgboost, precision_threshold)
    ceilings[name] = cv
    aucpr = [
        average_precision_score(y[fold_of == k], cv.scores[fold_of == k])
        for k in range(N_FOLDS)
    ]
    flagged = cv.flags == 1
    tp = int((flagged & (y == 1)).sum())
    print(
        f"{name}: AUC-PR {np.mean(aucpr):.4f} +- {np.std(aucpr):.4f}, "
        f"flagged {int(flagged.sum()):,}, precision {tp / flagged.sum():.4f}, "
        f"recall {tp / y.sum():.4f}"
    )

summary: AUC-PR 0.2821 +- 0.0195, flagged 1,778, precision 0.7002, recall 0.1962


summary and rows: AUC-PR 0.3026 +- 0.0242, flagged 1,865, precision 0.7013, recall 0.2061
CPU times: total: 10min 7s
Wall time: 44 s


## 3. The ceiling on the 203 accounts, against the LLM alone

Correct decisions per group in K1: the classifier's flag against the label, and each LLM model's decision. Then one number without a threshold on the same accounts: AUC-ROC, with the LLM's score taken as its confidence when it says suspicious and one minus its confidence otherwise. The sample has equal groups, so this AUC describes this sample only.

In [4]:
sample = pd.read_parquet(DATA_INTERIM / "llm_experiment_cases.parquet")["group"]
answers = pd.read_parquet(DATA_INTERIM / "pipeline" / "results.parquet")
alone = answers[answers["condition"] == K1].set_index(["model", "account"])
assert alone["decision"].notna().all()
truth = y[sample.index]
right = {}
auc = {}
for name, cv in ceilings.items():
    right[f"ceiling, {name}"] = cv.flags[sample.index] == truth
    auc[f"ceiling, {name}"] = roc_auc_score(truth, cv.scores[sample.index])
for model in ["qwen3.5:9b-q4_K_M", "gemma4:latest"]:
    mine = alone.loc[model].loc[sample.index]
    says = mine["decision"] == SUSPICIOUS
    right[model] = says == (truth == 1)
    auc[model] = roc_auc_score(
        truth, np.where(says, mine["confidence"], 1 - mine["confidence"])
    )
per_group = pd.DataFrame(right).groupby(sample).sum().reindex(GROUP_NAMES)
print("correct of 29 per group")
print(per_group.to_string())
print()
print("AUC-ROC on the 203 accounts")
print(pd.Series(auc).round(3).to_string())

correct of 29 per group


                             ceiling, summary  ceiling, summary and rows  qwen3.5:9b-q4_K_M  gemma4:latest
group                                                                                                     
strong, model right                        15                         15                 26             21
strong, model wrong                        22                         22                 10             16
uncertain, model right                      4                          7                 19             12
uncertain, model wrong                     28                         29                  5             14
missed, in the pattern file                 0                          0                 16             15
missed, outside the file                    0                          0                 21             15
not flagged, clean                         29                         29                  9             17

AUC-ROC on the 203 accounts
ceiling,

## 4. Saved for the independent check

The features and the out-of-fold scores of every account, in `data/interim`, which is not in the repository. Notebook 13 adds to these features.

In [5]:
X.to_parquet(DATA_INTERIM / "ceiling_features.parquet")
pd.DataFrame(
    {
        f"{kind} {name}": getattr(cv, kind)
        for name, cv in ceilings.items()
        for kind in ["scores", "flags"]
    }
).to_parquet(DATA_INTERIM / "ceiling_scores.parquet")
print("saved")

saved
